# Lab 01 Solution: Train a Discrete Diffusion Model on TinyStories

This solution uses MDLM with a cosine masking schedule.

In [ ]:
%pip install torch numpy matplotlib datasets tqdm --quiet

In [ ]:
import sys
sys.path.insert(0, '../../..')
sys.path.insert(0, '../lesson03-d3pm-from-scratch')
sys.path.insert(0, '../lesson04-mdlm')
sys.path.insert(0, '../lesson05-training-and-sampling')

import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm
from shared.utils.seed import set_seed
from shared.utils.device import get_device
from shared.datasets.text import SimpleTokenizer, TextDataset, load_text_dataset

set_seed(42)
device = get_device()
print(f"Using device: {device}")

## Step 1: Load and Prepare Data

In [ ]:
texts = load_text_dataset("tinystories", max_samples=5000)
tokenizer = SimpleTokenizer(texts, level="char")

SEQ_LEN = 64
dataset = TextDataset(texts, tokenizer, seq_len=SEQ_LEN)

BATCH_SIZE = 64
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Dataset size: {len(dataset)}")
print(f"Sample: '{tokenizer.decode(dataset[0].tolist())}'")

## Step 2: Build the Model

In [ ]:
from src.mdlm import MDLMDenoiser, MDLM

denoiser = MDLMDenoiser(
    vocab_size=tokenizer.vocab_size,
    d_model=128,
    n_heads=4,
    n_layers=4,
    max_seq_len=SEQ_LEN,
    dropout=0.1,
).to(device)

model = MDLM(
    denoiser=denoiser,
    vocab_size=tokenizer.vocab_size,
    mask_token_id=tokenizer.mask_id,
    num_timesteps=100,
    schedule_type="cosine",
    device=device,
)

n_params = sum(p.numel() for p in denoiser.parameters())
print(f"Model parameters: {n_params:,}")

## Step 3: Training Loop

In [ ]:
from src.training_utils import get_cosine_schedule_with_warmup

optimizer = torch.optim.Adam(denoiser.parameters(), lr=3e-4)

NUM_EPOCHS = 30
total_steps = NUM_EPOCHS * len(dataloader)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)

In [ ]:
losses = []

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    n_batches = 0
    
    for batch in dataloader:
        batch = batch.to(device)
        
        loss = model.train_loss(batch)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(denoiser.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Loss: {avg_loss:.4f}")

print(f"Final loss: {losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 4: Generate Samples

In [ ]:
generated_samples = model.sample(batch_size=10, seq_len=SEQ_LEN, temperature=0.8)
generated_texts = [tokenizer.decode(s.cpu().tolist()) for s in generated_samples]

print("Generated stories:")
for i, text in enumerate(generated_texts):
    print(f"\n[{i+1}] '{text}'")

## Step 5: Evaluate Quality

In [ ]:
from src.training_utils import compute_perplexity_proxy

eval_batch = next(iter(dataloader)).to(device)
ppl_proxy = compute_perplexity_proxy(
    denoiser, eval_batch, tokenizer.mask_id, device=device
)

print(f"Perplexity proxy: {ppl_proxy:.4f}")
assert ppl_proxy < 3.5, f"Perplexity proxy {ppl_proxy:.4f} exceeds threshold 3.5"
print("PASSED: Perplexity proxy is below threshold.")

In [ ]:
# Quality check
common_words = {'the', 'a', 'is', 'and', 'in', 'on', 'to', 'it', 'was', 'he', 'she'}
n_good = sum(
    1 for text in generated_texts
    if any(word in text.lower() for word in common_words)
)

print(f"Samples with English words: {n_good}/10")
assert n_good >= 5, f"Only {n_good} good samples. Try lower temperature or more training."
print("PASSED: Sufficient quality.")
print("\nLab complete!")